# Step 3: Test non-representative sample of hard and soft cases

This notebook builds a curated non-representative test set for refining the annotation instructions. The goal is to create an informative set of `n = 500` with hard cases, positive and negative crime-frame cases, `NO_CRIME_FRAME` cases and unseen cases.

The workflow is:

1. Load the outputs from Step 2.1 and Step 2.2.
2. Build a curated Step 3 test set with hard cases, `CRIME_FRAME_NEG`, `CRIME_FRAME_POS`, `NO_CRIME_FRAME` cases and unseen cases using keyword search.
3. Export a clean file for human annotation.
4. Import human labels and calculate F1 scores and Cohen's kappa.
5. Optionally test few-shot prompting on the same test set.

## Step 3.0: Imports and file paths

This section defines the file paths used in Step 3. It expects that Step 2 created the following CSV files:

- `results/crime_frame_neg.csv`
- `results/crime_frame_pos.csv`
- `results/unclear_frame.csv`
- `results/hard_cases.csv`
- one or more full Step 2.1 result files named like `annotation_step2_1_*.csv`


In [13]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, cohen_kappa_score

# ── Config ───────────────────────────────────────────────────────────────
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Input files from Step 2
CRIME_FRAME_NEG = RESULTS_DIR / "crime_frame_neg.csv"
CRIME_FRAME_POS = RESULTS_DIR / "crime_frame_pos.csv"
UNCLEAR_FRAME = RESULTS_DIR / "unclear_frame.csv"
HARD_CASES = RESULTS_DIR / "hard_cases.csv"

# Step 3 output files
STEP3_TESTSET_WITH_MODEL_LABELS = RESULTS_DIR / "step3_testset_500_internal_with_model_info.csv"
STEP3_TESTSET_FOR_HUMAN = RESULTS_DIR / "step3_testset_500_for_human_annotation.csv"
STEP3_HUMAN_COMPLETED = RESULTS_DIR / "step3_testset_500_human_completed.csv"
STEP3_EVALUATION = RESULTS_DIR / "step3_evaluation_summary.csv"
STEP3_FEWSHOT_RESULTS = RESULTS_DIR / "step3_fewshot_results.csv"

# Test set size and composition -> can be changed! Maybe also include should take all the crime_pos cases
N_TESTSET = 500
TESTSET_SEED = 123

PCT_HARD = 0.30
PCT_CRIME = 0.40
PCT_NO_CRIME = 0.25
PCT_KEYWORD = 0.05

N_HARD = int(N_TESTSET * PCT_HARD)
N_CRIME = int(N_TESTSET * PCT_CRIME)
N_NO_CRIME = int(N_TESTSET * PCT_NO_CRIME)
N_KEYWORD = N_TESTSET - N_HARD - N_CRIME - N_NO_CRIME

VALID_LABELS = ["NO_CRIME_FRAME", "CRIME_FRAME_NEG", "CRIME_FRAME_POS"]
# ─────────────────────────────────────────────────────────────────────────

print("Step 3 setup complete.")
print(f"Target testset size: {N_TESTSET}")
print(f"Hard cases: {N_HARD}")
print(f"Crime frame cases: {N_CRIME}")
print(f"No crime cases: {N_NO_CRIME}")
print(f"Keyword unseen cases: {N_KEYWORD}")


Step 3 setup complete.
Target testset size: 500
Hard cases: 150
Crime frame cases: 200
No crime cases: 125
Keyword unseen cases: 25


## Step 3.1: Load the candidate files from Step 2

We load the separate case files from Step 2. Empty or missing files are handled safely so the notebook can still run if one category has not appeared yet.


In [14]:
def read_optional_csv(path):
    if path.exists():
        df_loaded = pd.read_csv(path)
        df_loaded["source_file"] = path.name
        print(f"Loaded {path}: {len(df_loaded)} rows")
        return df_loaded
    else:
        print(f"Missing {path}: using empty dataframe")
        return pd.DataFrame()

crime_neg = read_optional_csv(CRIME_FRAME_NEG)
crime_pos = read_optional_csv(CRIME_FRAME_POS)
unclear = read_optional_csv(UNCLEAR_FRAME)
hard_cases = read_optional_csv(HARD_CASES)

print("\nLoaded candidate files:")
print(f"CRIME_FRAME_NEG: {len(crime_neg)}")
print(f"CRIME_FRAME_POS: {len(crime_pos)}")
print(f"UNCLEAR: {len(unclear)}")
print(f"HARD_CASES: {len(hard_cases)}")


Loaded results/crime_frame_neg.csv: 839 rows
Loaded results/crime_frame_pos.csv: 17 rows
Loaded results/unclear_frame.csv: 0 rows
Loaded results/hard_cases.csv: 323 rows

Loaded candidate files:
CRIME_FRAME_NEG: 839
CRIME_FRAME_POS: 17
UNCLEAR: 0
HARD_CASES: 323


## Step 3.2: Load full Step 2.1 outputs for `NO_CRIME_FRAME` cases


In [15]:
step2_1_files = sorted(RESULTS_DIR.glob("annotation_step2_1_*.csv"))

if not step2_1_files:
    raise FileNotFoundError(
        "No full Step 2.1 files found. Expected files like results/annotation_step2_1_*.csv"
    )

step2_1_all = []

for file in step2_1_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    step2_1_all.append(temp)

step2_1_all = pd.concat(step2_1_all, ignore_index=True)
step2_1_all = step2_1_all.drop_duplicates(subset=["article_id", "par_index"])

no_crime_candidates = step2_1_all[
    step2_1_all["label"] == "NO_CRIME_FRAME"
].copy()

print(f"Loaded {len(step2_1_files)} Step 2.1 files")
print(f"Unique Step 2.1 rows: {len(step2_1_all)}")
print(f"NO_CRIME_FRAME candidates: {len(no_crime_candidates)}")


Loaded 2 Step 2.1 files
Unique Step 2.1 rows: 19925
NO_CRIME_FRAME candidates: 19076


## Step 3.3: Standardize candidates and build the 500 row test set

The test set includes all available hard cases, unclear cases, negative crime-frame cases, and positive crime-frame cases. The remaining rows are filled with `NO_CRIME_FRAME` cases.

If the interesting cases exceed 500 rows a mixed sample is used.


In [16]:
def standardize_cases(df_cases, case_source, model_label_col="label"):
    if df_cases is None or len(df_cases) == 0:
        return pd.DataFrame(columns=[
            "article_id", "par_index", "group", "text_block_en", "model_label", "case_source"
        ])

    df_cases = df_cases.copy()

    if "text_block_en" not in df_cases.columns and "text_block" in df_cases.columns:
        df_cases["text_block_en"] = df_cases["text_block"]

    if model_label_col not in df_cases.columns:
        if "final_label" in df_cases.columns:
            model_label_col = "final_label"
        else:
            df_cases["model_label"] = ""
            model_label_col = "model_label"

    keep_cols = ["article_id", "par_index", "group", "text_block_en"]
    existing_keep_cols = [c for c in keep_cols if c in df_cases.columns]

    out = df_cases[existing_keep_cols].copy()
    out["model_label"] = df_cases[model_label_col].values
    out["case_source"] = case_source

    for extra_col in ["agreement", "step2_1_label", "final_label", "source_file"]:
        if extra_col in df_cases.columns:
            out[extra_col] = df_cases[extra_col].values

    return out


neg_cases = standardize_cases(crime_neg, "crime_frame_neg", model_label_col="label")
pos_cases = standardize_cases(crime_pos, "crime_frame_pos", model_label_col="label")
unclear_cases = standardize_cases(unclear, "unclear", model_label_col="label")
hard_case_rows = standardize_cases(hard_cases, "hard_case", model_label_col="final_label")
no_crime_rows = standardize_cases(no_crime_candidates, "no_crime", model_label_col="label")


# Combine all candidates without changing the original CSVs
candidate_sets = [
    hard_case_rows,
    neg_cases,
    pos_cases,
    unclear_cases,
    no_crime_rows
]

all_candidates = pd.concat(candidate_sets, ignore_index=True)

source_priority = {
    "hard_case": 1,
    "crime_frame_neg": 2,
    "crime_frame_pos": 2,
    "unclear": 3,
    "no_crime": 4
}

all_candidates["source_priority"] = all_candidates["case_source"].map(source_priority)
all_candidates = all_candidates.sort_values("source_priority")

all_candidates_unique = all_candidates.drop_duplicates(
    subset=["article_id", "par_index"],
    keep="first"
).copy()

print("Before deduplication")
print(all_candidates["case_source"].value_counts())

print("\nAfter deduplication")
print(all_candidates_unique["case_source"].value_counts())


# Pools for percentage based sampling
hard_pool = all_candidates_unique[
    all_candidates_unique["case_source"] == "hard_case"
].copy()

crime_pool = all_candidates_unique[
    all_candidates_unique["case_source"].isin(["crime_frame_neg", "crime_frame_pos"])
].copy()

no_crime_pool = all_candidates_unique[
    all_candidates_unique["case_source"] == "no_crime"
].copy()


# Keyword unseen cases
df_full = pd.read_csv("dataset/all_multi_paragraphs_2022_2026.csv")

security_keywords = [
    "diebstahl", "raub", "gewalt", "polizei", "ermittlung",
    "verdächtig", "tatverdächtig", "festnahme", "terror",
    "extremismus", "bedrohung", "gefahr", "sicherheit",
    "illegal", "abschiebung", "grenze", "angriff",
    "anschlag", "mord", "körperverletzung", "kriminalität",
    "straftat", "schutz", "prävention"
]

pattern = "|".join(security_keywords)

keyword_rows = df_full[
    df_full["text_block"].str.lower().str.contains(pattern, na=False)
].copy()

keyword_rows = standardize_cases(keyword_rows, "keyword_unseen", model_label_col="label")


def row_keys(data):
    return data["article_id"].astype(str) + "_" + data["par_index"].astype(str)


def remove_used(data, used_keys):
    if len(data) == 0:
        return data

    return data[~row_keys(data).isin(used_keys)].copy()


def sample_n(data, n, seed):
    if len(data) == 0:
        return data

    return data.sample(n=min(n, len(data)), random_state=seed).copy()


used_keys = set()

hard_sample = sample_n(hard_pool, N_HARD, TESTSET_SEED)
used_keys.update(row_keys(hard_sample))

crime_pool = remove_used(crime_pool, used_keys)
crime_sample = sample_n(crime_pool, N_CRIME, TESTSET_SEED)
used_keys.update(row_keys(crime_sample))

no_crime_pool = remove_used(no_crime_pool, used_keys)
no_crime_sample = sample_n(no_crime_pool, N_NO_CRIME, TESTSET_SEED)
used_keys.update(row_keys(no_crime_sample))

keyword_rows = remove_used(keyword_rows, used_keys)
keyword_sample = sample_n(keyword_rows, N_KEYWORD, TESTSET_SEED)
used_keys.update(row_keys(keyword_sample))


step3_testset = pd.concat(
    [hard_sample, crime_sample, no_crime_sample, keyword_sample],
    ignore_index=True
).drop_duplicates(subset=["article_id", "par_index"]).reset_index(drop=True)

step3_testset["testset_id"] = range(1, len(step3_testset) + 1)

cols = ["testset_id"] + [c for c in step3_testset.columns if c != "testset_id"]
step3_testset = step3_testset[cols]

step3_testset.to_csv(STEP3_TESTSET_WITH_MODEL_LABELS, index=False)

print(f"Step 3 internal test set saved to {STEP3_TESTSET_WITH_MODEL_LABELS}")
print(f"Rows in Step 3 test set: {len(step3_testset)}")

print("\nCase source distribution:")
print(step3_testset["case_source"].value_counts())

print("\nModel label distribution:")
print(step3_testset["model_label"].value_counts())


Before deduplication
case_source
no_crime           19076
crime_frame_neg      839
hard_case            323
crime_frame_pos       17
Name: count, dtype: int64

After deduplication
case_source
no_crime           19042
crime_frame_neg      555
hard_case            323
crime_frame_pos        5
Name: count, dtype: int64
Step 3 internal test set saved to results/step3_testset_500_internal_with_model_info.csv
Rows in Step 3 test set: 500

Case source distribution:
case_source
crime_frame_neg    197
hard_case          150
no_crime           125
keyword_unseen      25
crime_frame_pos      3
Name: count, dtype: int64

Model label distribution:
model_label
CRIME_FRAME_NEG    298
NO_CRIME_FRAME     171
                    25
CRIME_FRAME_POS      6
Name: count, dtype: int64


## Step 3.4: Export file for human annotation

This file is meant for manual annotation. It hides the model labels so that human annotators are not biased by the model output.

Fill the `human_label` column manually with exactly one of:

- `NO_CRIME_FRAME`
- `CRIME_FRAME_NEG`
- `CRIME_FRAME_POS`

Optional: use `human_notes` for short comments about difficult cases.


In [17]:
human_export = step3_testset[
    ["testset_id", "article_id", "par_index", "group", "text_block_en"]
].copy()

human_export["label_robin"] = ""
human_export["label_hamid"] = ""
human_export["label_marmee"] = ""

human_export["notes_robin"] = ""
human_export["notes_hamid"] = ""
human_export["notes_marmee"] = ""

human_export.to_csv(STEP3_TESTSET_FOR_HUMAN, index=False)

print(f"Human annotation file saved to {STEP3_TESTSET_FOR_HUMAN}")
human_export.head()

Human annotation file saved to results/step3_testset_500_for_human_annotation.csv


,testset_id,article_id,par_index,group,text_block_en,label_robin,label_hamid,label_marmee,notes_robin,notes_hamid,notes_marmee
0,1,OSZ-doc7k2ebrm9wrognjx5oq7,6,RUS,"Anno August Jagdfeld, der im Marien-Cottage in...",,,,,,
1,2,achse_added_fdcc00add3c941db5b52f21d5475839ec9...,16,REFUG,Bayern: [Gruppe 1] identifizieren\n\nBayern ha...,,,,,,
2,3,632712a23dd49b919c1a13db,5,REFUG,In Griechenland werden [Gruppe 1] als Handlang...,,,,,,
3,4,632712b03dd49b919c1f6623,6,JESID,Al-Kuraischi sprengte sich selbst in die Luft\...,,,,,,
4,5,Mmnews_added_c4d96bca88e842d7092f16699f62056e9...,3,REFUG,Der SPD-Vorsitzende Lars Klingbeil hält die Ei...,,,,,,


## Step 3.5: Import and check human labels and evaluate model performance
First we check how strongly the three human annotators agree with each other. Each row is annotated independently by Robin, Hamid, and Marmee.

A final human consensus label is created only when at least two annotators choose the same label. If all three annotators choose different labels, the row is flagged as a human hard case and should be discussed manually.

After human annotation is finished, save the completed file as:

`results/step3_testset_500_human_completed.csv`

The file must include:

- `testset_id`
- `label_robin`
- `label_hamid`
- `label_marmee`

After creating the human consensus labels, we compare the model labels against the consensus labels. We report the confusion matrix, F1 scores, and Cohen's kappa.


In [18]:
# ──Check human annotation agreement ──────────────────────────

human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)

annotator_cols = ["label_robin", "label_hamid", "label_marmee"]

# Check for invalid labels
for col in annotator_cols:
    invalid = human_completed[
        human_completed[col].notna() &
        ~human_completed[col].isin(VALID_LABELS)
    ]

    if len(invalid) > 0:
        print(f"Warning: {col} has invalid labels:")
        print(invalid[["testset_id", col]].head(20))

def get_majority_label(row):
    labels = [row[col] for col in annotator_cols if pd.notna(row[col])]

    if len(labels) == 0:
        return np.nan

    counts = pd.Series(labels).value_counts()

    # At least 2 annotators agree
    if counts.iloc[0] >= 2:
        return counts.index[0]

    # All annotators gave different labels
    return "NO_HUMAN_CONSENSUS"

human_completed["human_consensus_label"] = human_completed.apply(get_majority_label, axis=1)

# Count how many annotators agreed
def get_human_agreement(row):
    labels = [row[col] for col in annotator_cols if pd.notna(row[col])]

    if len(labels) == 0:
        return np.nan

    counts = pd.Series(labels).value_counts()
    return counts.iloc[0] / len(labels)

human_completed["human_agreement"] = human_completed.apply(get_human_agreement, axis=1)

# Flag rows where all three annotators disagree
human_hard_cases = human_completed[
    human_completed["human_consensus_label"] == "NO_HUMAN_CONSENSUS"
].copy()

print("=== Human annotation agreement ===")
print(human_completed["human_agreement"].describe().round(2))

print("\nConsensus label distribution:")
print(human_completed["human_consensus_label"].value_counts(dropna=False))

print(f"\nHuman hard cases, all three annotators disagree: {len(human_hard_cases)}")

# Save human hard cases for manual discussion
STEP3_HUMAN_HARD_CASES = RESULTS_DIR / "step3_human_hard_cases_for_discussion.csv"

human_hard_cases.to_csv(STEP3_HUMAN_HARD_CASES, index=False)

print(f"Saved human hard cases to {STEP3_HUMAN_HARD_CASES}")

FileNotFoundError: [Errno 2] No such file or directory: 'results/step3_testset_500_human_completed.csv'

In [ ]:
# ── Evaluate model against human consensus labels ─────────────

merged = step3_testset.merge(
    human_completed[["testset_id", "human_consensus_label", "human_agreement"]],
    on="testset_id",
    how="left"
)

eval_data = merged[
    merged["human_consensus_label"].isin(VALID_LABELS)
].copy()

print(f"Rows used for model evaluation: {len(eval_data)}")
print(f"Rows excluded because of no human consensus: {len(merged) - len(eval_data)}")

print("\nHuman consensus label distribution:")
print(eval_data["human_consensus_label"].value_counts())

print("\nModel label distribution:")
print(eval_data["model_label"].value_counts())

print("\nConfusion matrix:")
print(confusion_matrix(
    eval_data["human_consensus_label"],
    eval_data["model_label"],
    labels=VALID_LABELS
))

print("\nClassification report:")
print(classification_report(
    eval_data["human_consensus_label"],
    eval_data["model_label"],
    labels=VALID_LABELS
))

kappa = cohen_kappa_score(
    eval_data["human_consensus_label"],
    eval_data["model_label"],
    labels=VALID_LABELS
)

macro_f1 = f1_score(
    eval_data["human_consensus_label"],
    eval_data["model_label"],
    labels=VALID_LABELS,
    average="macro"
)

weighted_f1 = f1_score(
    eval_data["human_consensus_label"],
    eval_data["model_label"],
    labels=VALID_LABELS,
    average="weighted"
)

print(f"\nCohen's kappa: {kappa:.3f}")
print(f"Macro F1: {macro_f1:.3f}")
print(f"Weighted F1: {weighted_f1:.3f}")


Before deduplication
case_source
no_crime           19076
crime_frame_neg      839
hard_case            238
crime_frame_pos       17
Name: count, dtype: int64

After deduplication
case_source
no_crime           19047
crime_frame_neg      631
hard_case            238
crime_frame_pos        9
Name: count, dtype: int64


ParserError: Error tokenizing data. C error: Calling read(nbytes) on source failed. Try engine='python'.

## Step 3.6: Inspect model errors

This section helps identify where the instruction needs improvement. These rows are especially useful for revising the codebook or writing few-shot examples.


In [ ]:
if not STEP3_HUMAN_COMPLETED.exists():
    print("Human labels not available yet.")
else:
    errors = eval_data[
        eval_data["human_consensus_label"] != eval_data["model_label"]
    ].copy()

    print(f"Errors: {len(errors)}")

    display(errors[
        [
            "testset_id",
            "case_source",
            "group",
            "human_consensus_label",
            "model_label",
            "text_block_en"
        ]
    ].head(30))


# Step 3.2: Few-shot examples

Few-shot examples should be manually curated. Ideally, do not copy cases from the Step 3 test set into the few-shot prompt, because that would bias the evaluation. Use invented examples or separate training examples.

Good few-shot examples should include:

- at least one clear case per label
- hard boundary cases, especially victim versus perpetrator
- cases where crime is mentioned but not linked to the group
- cases where terrorism or violence is legitimized by a group
- cases where security or prevention is framed positively

The order should be mixed so the model does not learn a simple label sequence.


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "Beispieltext hier einfügen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Keine explizite Verknüpfung der Gruppe mit Kriminalität oder Sicherheit."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird explizit als Quelle von Gefahr, Kriminalität oder Sicherheitsbedrohung dargestellt."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Der Text rahmt Sicherheit, Prävention oder Schutz im Zusammenhang mit der Gruppe positiv."
    },
]

def format_few_shot_examples(examples):
    formatted = []
    for ex in examples:
        formatted.append(
            "Text:
"
            + ex["text"]
            + "

Label: "
            + ex["label"]
            + "
Explanation: "
            + ex["explanation"]
        )
    return "

---

".join(formatted)

few_shot_block = format_few_shot_examples(FEW_SHOT_EXAMPLES)
print(few_shot_block)


## Step 3.7: Optional few-shot annotation function

This version adds the few-shot examples to the prompt before the paragraph to annotate. It uses the same output format as the earlier annotation function.

Run this only after you have added real few-shot examples above and after the API variables are available from your setup.


In [ ]:
def annotate_few_shot(text, instructions, reminder, few_shot_examples, temperature=0.0):
    few_shot_text = format_few_shot_examples(few_shot_examples)

    prompt = (
        f"{instructions}

"
        f"Hier sind Beispiele für die Annotation:

{few_shot_text}

"
        f"Jetzt annotieren Sie diesen Absatz:

{text}

"
        f"{reminder}

"
        "Respond in this format exactly:
"
        "Label: <NO_CRIME_FRAME or CRIME_FRAME_NEG or CRIME_FRAME_POS>
"
        "Explanation: <one sentence explaining your choice>"
    )

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)

    return "ERROR"


## Step 3.8: Optional few-shot run on the Step 3 test set

This runs the few-shot prompt on the Step 3 test set and saves the result. It can then be compared against the human labels using the same metrics as above.


In [ ]:
RUN_FEW_SHOT = False

if RUN_FEW_SHOT:
    fewshot_results = []

    for i, row in step3_testset.iterrows():
        raw = annotate_few_shot(
            str(row["text_block_en"]),
            instructions,
            reminder,
            FEW_SHOT_EXAMPLES,
            temperature=0.0
        )
        label = parse_label(raw)
        explanation = parse_explanation(raw)

        fewshot_results.append({
            "testset_id": row["testset_id"],
            "article_id": row["article_id"],
            "par_index": row["par_index"],
            "group": row["group"],
            "text_block_en": row["text_block_en"],
            "fewshot_raw_output": raw,
            "fewshot_label": label,
            "fewshot_explanation": explanation,
        })

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/{len(step3_testset)}] done")

    fewshot_results = pd.DataFrame(fewshot_results)
    fewshot_results.to_csv(STEP3_FEWSHOT_RESULTS, index=False)
    print(f"Few-shot results saved to {STEP3_FEWSHOT_RESULTS}")
else:
    print("RUN_FEW_SHOT is False. Set it to True when you want to run few-shot annotation.")


## Step 3.9: Optional evaluation of few-shot results

After few-shot annotation and human labels are available, this compares the few-shot labels to the human standard labels.


In [ ]:
if not STEP3_FEWSHOT_RESULTS.exists() or not STEP3_HUMAN_COMPLETED.exists():
    print("Few-shot results or human completed file not available yet.")
else:
    fewshot_results = pd.read_csv(STEP3_FEWSHOT_RESULTS)
    human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)

    few_eval = fewshot_results.merge(
        human_completed[["testset_id", "human_label"]],
        on="testset_id",
        how="left"
    )

    few_eval["human_label"] = few_eval["human_label"].astype(str).str.strip()
    few_eval_ready = few_eval[few_eval["human_label"].isin(VALID_LABELS)].copy()

    y_true = few_eval_ready["human_label"]
    y_pred = few_eval_ready["fewshot_label"]

    print("=== Few-shot Classification Report ===")
    print(classification_report(y_true, y_pred, labels=VALID_LABELS, zero_division=0))

    print("=== Few-shot Cohen's Kappa ===")
    print(cohen_kappa_score(y_true, y_pred, labels=VALID_LABELS))
